In [1]:
import pandas as pd
import datasets
from datasets import Image as Image_ds # change name because of similar PIL module
from datasets import Dataset
import os
import requests 
from PIL import Image
from tqdm import tqdm

/Users/au672746/Library/CloudStorage/OneDrive-Aarhusuniversitet/CHC/art-multimodal-benchmark/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
omniart = pd.read_csv(os.path.join('data', 'omniart_v3_datadump.csv'))

In [12]:
omniart['image_url'].iloc[0]

'https://img00.deviantart.net/839b/i/2015/127/9/a/70_amx_by_xynphix-dyxa31.jpg'

In [3]:
omniart_paintings = omniart.query('general_type == "painting"')

In [14]:
omniart_paintings.columns

Index(['id', 'artwork_name', 'artist_full_name', 'artist_first_name',
       'artist_last_name', 'creation_year', 'century', 'source_url',
       'image_url', 'collection_origins', 'artwork_type', 'school',
       'original_id_in_collection', 'created_at', 'last_modified', 'omni_id',
       'created_by_id', 'general_type', 'geocoded', 'color_pallete',
       'dominant_color', 'palette_count'],
      dtype='object')

In [17]:
# save image thumbnail and fetch it using API
thumbnail = omniart_paintings['image_url'].iloc[0] # a 'thumbnail' is just a smaller version of the image (image native is around 10000x10000 pixels, takes forever to get)
img = Image.open(requests.get(thumbnail, stream = True).raw)

In [4]:
def download_image(url):
    '''
    Download image from thumbnail URL and encode it to HF feature
    '''
    feature = Image_ds(decode=False) 
    try:
        img = Image.open(requests.get(url, stream=True).raw) # stream=True enables to download image in chunks and not all at once

        # encode the PIL images to image feature
        image_encoded = feature.encode_example(img)

        return image_encoded

    except Exception as e:
        print(f"Error processing image: {e}")
        return pd.NA

In [5]:
omni_subset = omniart_paintings.iloc[:20].reset_index(drop=True)

In [6]:
images = [download_image(url) for url in tqdm(omni_subset['image_url'], desc= 'Downloading images')]

In [7]:
omni_subset['image'] = images

In [8]:
ds = Dataset.from_pandas(omni_subset).cast_column('image', Image_ds(decode=True)) # decode back to PIL image from raw bytes

# save HF dataset to data folder
ds.save_to_disk(os.path.join('data', 'omni_test'))

Saving the dataset (1/1 shards): 100%|██████████| 20/20 [00:00<00:00, 2009.44 examples/s]


In [9]:
import torch
from transformers import pipeline

In [10]:
import diffusers

In [11]:
pipe = pipeline(task="image-feature-extraction", model_name="stabilityai/stable-diffusion-3.5-large", pool=True)

No model was supplied, defaulted to google/vit-base-patch16-224 and revision 3f49326 (https://huggingface.co/google/vit-base-patch16-224).
Using a pipeline without specifying a model name and revision in production is not recommended.
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
from diffusers import AutoencoderKL

# 1. Load the autoencoder model which will be used to decode the latents into image space. 
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae")

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


In [ ]:
import numpy as np
from torchvision import transforms
# Convert PIL image to tensor and add batch dimension
image = ds[0]['image']  # likely a PIL image

preprocess = transforms.Compose([
    transforms.ToTensor(),                     # Converts to (C, H, W) and normalizes to [0,1]
    transforms.Resize((512, 512)),             # Resize to match model input (optional, adjust as needed)
])

tensor = preprocess(image).unsqueeze(0)        # (1, C, H, W) — adds batch dimension

# Move to same device as the model
tensor = tensor.to(vae.device)

# Run through VAE
output = vae(tensor).sample # GENERATES NEW IMAGE, NOT COMPRESSED EMBEDDING

In [33]:
output.detach().numpy().shape

(1, 3, 512, 512)

In [ ]:
ou

()